# Семинар 10. Свой класс и его состояние

Сегодня напишем пять отдельных классов. У каждого есть состояние и небольшой набор операций: переместить точку, увеличить счётчик, изменить список воспроизведения, накопить измерения или пройти вопросы теста. Проверим не только правильный результат, но и то, что ошибка не оставляет объект в неправильном состоянии.

## Цели

- создавать экземпляры с собственной инициализацией и атрибутами;
- связывать изменение состояния с вызовом метода;
- предоставлять чтение через свойства без публичного присваивания;
- проверять ограничения до изменения состояния;
- отделять внутреннюю коллекцию от входного списка и выданной копии;
- проверять независимость экземпляров одного класса.

## Перед началом

Нужны функции, ссылки и изменяемые объекты, исключения и [лекция 10](lecture.ipynb). Наследование, `dataclass`, собственные контекстные менеджеры и взаимодействие нескольких классов не требуются. Все задания работают на Python 3.14 без внешних файлов и пакетов.

В заготовках стоит `pass`. Сначала напишите класс, затем выполните ячейку с проверкой и уберите комментарий перед вызовом `check_task_N()`. Исходный `Run All` проходит, потому что проверки ещё не вызываются; это не означает, что задачи решены.

Типы аргументов соответствуют аннотациям. Не проверяйте типы и ограничения, которые прямо гарантированы. Проверяйте перечисленное некорректное содержимое. Методы ничего не печатают. Если возврат не указан отдельно, метод возвращает `None`.

Публичные свойства, объявленные только для чтения, не должны иметь setter: присваивание им извне приводит к `AttributeError`. Не нужно защищать класс от намеренного изменения внутренних атрибутов с `_`.

## Задание 1. Подвижная точка

Напишите `Point(x, y)` с публичными атрибутами `x` и `y`. Метод `move(dx, dy)` прибавляет смещения к координатам и возвращает `None`.

Две точки не должны делить координаты. Ещё одно имя той же точки обращается к тому же экземпляру. Например, после `alias = point` вызов `alias.move(...)` перемещает и объект, доступный через `point`.

Все аргументы — целые числа по модулю не больше 10⁹; число перемещений до 100 000. Ограничений на полученные координаты нет. Время и память создания и одного перемещения — `O(1)`.

In [ ]:
class Point:
    def __init__(self, x: int, y: int) -> None:
        pass

    def move(self, dx: int, dy: int) -> None:
        pass

In [ ]:
def check_task_1():
    point = Point(2, -1)
    other = Point(2, -1)
    assert point is not other
    assert point.move(3, 4) is None
    assert (point.x, point.y) == (5, 3)
    assert (other.x, other.y) == (2, -1)
    alias = point
    assert alias.move(-5, 0) is None
    assert (point.x, point.y) == (0, 3)
    print("Задание 1: проверки пройдены")

# check_task_1()

## Задание 2. Счётчик до границы

Напишите `BoundedCounter(limit, start=0)`.

- `limit` должен быть неотрицательным, `start` — от 0 до `limit` включительно. Иначе конструктор поднимает `ValueError`.
- Свойства `value` и `limit` доступны только для чтения.
- `increment(amount=1)` увеличивает значение на положительное `amount`. Если прибавка неположительна или новая величина превысила бы лимит, поднимается `ValueError`, а состояние остаётся прежним.
- `reset()` устанавливает значение 0, даже если начальное значение было другим.

Все числа целые, по модулю не больше 10⁹; до 100 000 вызовов. Создание и каждый метод — `O(1)` времени и памяти.

In [ ]:
class BoundedCounter:
    def __init__(self, limit: int, start: int = 0) -> None:
        pass

    @property
    def value(self) -> int:
        pass

    @property
    def limit(self) -> int:
        pass

    def increment(self, amount: int = 1) -> None:
        pass

    def reset(self) -> None:
        pass

In [ ]:
def check_task_2():
    counter = BoundedCounter(5, start=2)
    other = BoundedCounter(5)
    assert counter.increment() is None
    assert counter.value == 3 and other.value == 0
    assert counter.increment(2) is None
    assert counter.value == 5
    for amount in [1, 0, -1]:
        try:
            counter.increment(amount)
        except ValueError:
            pass
        else:
            raise AssertionError("Недопустимую прибавку нужно отклонить")
        assert counter.value == 5
    assert counter.reset() is None
    assert counter.value == 0 and counter.limit == 5
    assert BoundedCounter(0).value == 0
    for args in [(-1,), (5, -1), (5, 6)]:
        try:
            BoundedCounter(*args)
        except ValueError:
            pass
        else:
            raise AssertionError("Недопустимое начальное состояние")
    for name in ["value", "limit"]:
        try:
            setattr(counter, name, 100)
        except AttributeError:
            pass
        else:
            raise AssertionError(f"{name} должно быть только для чтения")
    print("Задание 2: проверки пройдены")

# check_task_2()

## Задание 3. Список воспроизведения

Напишите `Playlist(tracks)`, который хранит названия треков в порядке добавления. Исходный список не должен становиться внутренним списком экземпляра.

Методы и свойства:

- `add(title)` добавляет название в конец;
- `remove_first()` извлекает и возвращает первое название; для пустого списка — `IndexError` без изменения состояния;
- `snapshot()` возвращает новый список текущих названий;
- свойство `count`, только для чтения, показывает число оставшихся треков.

Названия — произвольные строки, включая пустые; повторы разрешены и обозначают отдельные треки. Добавление в исходный список или изменение полученной копии не меняет экземпляр. Два экземпляра не делят внутренний список.

До 10 000 треков и до 10 000 операций; строки длиной до 100 символов. `add` — амортизированная `O(1)`, `count` — `O(1)`, `remove_first` и `snapshot` — не хуже `O(n)` по текущей длине. Здесь достаточно обычного списка.

In [ ]:
class Playlist:
    def __init__(self, tracks: list[str]) -> None:
        pass

    @property
    def count(self) -> int:
        pass

    def add(self, title: str) -> None:
        pass

    def remove_first(self) -> str:
        pass

    def snapshot(self) -> list[str]:
        pass

In [ ]:
def check_task_3():
    source = ["A", "A"]
    playlist = Playlist(source)
    other = Playlist([])
    source.append("извне")
    assert playlist.snapshot() == ["A", "A"]
    view = playlist.snapshot()
    view.clear()
    assert playlist.count == 2
    assert playlist.add("") is None
    assert playlist.remove_first() == "A"
    assert playlist.snapshot() == ["A", ""]
    assert playlist.remove_first() == "A"
    assert playlist.remove_first() == ""
    assert playlist.count == 0 and other.snapshot() == []
    try:
        playlist.remove_first()
    except IndexError:
        pass
    else:
        raise AssertionError("Пустой список должен дать IndexError")
    assert playlist.snapshot() == []
    assert playlist.snapshot() is not playlist.snapshot()
    print("Задание 3: проверки пройдены")

# check_task_3()

## Задание 4. Статистика измерений

Напишите `ReadingStats()`, изначально не содержащий измерений.

- `add(value)` учитывает одно целое измерение, в том числе нулевое или отрицательное.
- Свойства `count` и `total`, только для чтения, возвращают число измерений и их сумму.
- `mean()` возвращает среднее как `float`; пока измерений нет — `ValueError`.
- `clear()` возвращает объект в начальное состояние.

Хранить все измерения не требуется: дополнительная память экземпляра должна быть `O(1)`. До 100 000 измерений по модулю до 10⁹. Каждый метод и чтение свойства — `O(1)`.

In [ ]:
class ReadingStats:
    def __init__(self) -> None:
        pass

    @property
    def count(self) -> int:
        pass

    @property
    def total(self) -> int:
        pass

    def add(self, value: int) -> None:
        pass

    def mean(self) -> float:
        pass

    def clear(self) -> None:
        pass

In [ ]:
def check_task_4():
    stats = ReadingStats()
    other = ReadingStats()
    assert (stats.count, stats.total) == (0, 0)
    try:
        stats.mean()
    except ValueError:
        pass
    else:
        raise AssertionError("У пустого набора нет среднего")
    assert stats.add(0) is None
    stats.add(-3)
    stats.add(6)
    assert (stats.count, stats.total) == (3, 3)
    assert stats.mean() == 1.0 and type(stats.mean()) is float
    assert (other.count, other.total) == (0, 0)
    assert stats.clear() is None
    assert (stats.count, stats.total) == (0, 0)
    stats.add(4)
    assert stats.mean() == 4.0
    print("Задание 4: проверки пройдены")

# check_task_4()

## Задание 5. Прохождение теста

Напишите `QuizSession(question_count)` для теста с заданным числом вопросов. В начале ответов нет.

- Отрицательное число вопросов — `ValueError` при создании. Нулевое допустимо.
- `record_answer(correct)` принимает `bool` и учитывает очередной ответ: число ответов увеличивается на 1, число верных — только для `True`.
- Когда все вопросы уже отвечены, ещё один вызов `record_answer` поднимает `RuntimeError` без изменения состояния.
- Свойства `answered`, `score` и `remaining` доступны только для чтения.
- `result()` возвращает `(верных, всего_вопросов)`, только если тест закончен; иначе — `RuntimeError`.

Для теста из нуля вопросов результат сразу равен `(0, 0)`. До 100 000 вопросов; число вызовов не больше 200 000. Каждый метод и свойство — `O(1)` времени, состояние занимает `O(1)` памяти.

In [ ]:
class QuizSession:
    def __init__(self, question_count: int) -> None:
        pass

    @property
    def answered(self) -> int:
        pass

    @property
    def score(self) -> int:
        pass

    @property
    def remaining(self) -> int:
        pass

    def record_answer(self, correct: bool) -> None:
        pass

    def result(self) -> tuple[int, int]:
        pass

In [ ]:
def check_task_5():
    session = QuizSession(2)
    other = QuizSession(1)
    try:
        session.result()
    except RuntimeError:
        pass
    else:
        raise AssertionError("Тест ещё не закончен")
    assert session.record_answer(False) is None
    assert (session.answered, session.score, session.remaining) == (1, 0, 1)
    session.record_answer(True)
    assert session.result() == (1, 2)
    assert (other.answered, other.score, other.remaining) == (0, 0, 1)
    try:
        session.record_answer(True)
    except RuntimeError:
        pass
    else:
        raise AssertionError("Лишний ответ должен быть отклонён")
    assert (session.answered, session.score, session.remaining) == (2, 1, 0)
    assert QuizSession(0).result() == (0, 0)
    try:
        QuizSession(-1)
    except ValueError:
        pass
    else:
        raise AssertionError("Число вопросов не должно быть отрицательным")
    print("Задание 5: проверки пройдены")

# check_task_5()

## Обсуждение

1. Почему атрибуты точки принадлежат экземпляру, хотя метод `move` определён один раз в классе?
2. Чем отличаются `other = Point(...)` и `other = point`?
3. Почему проверки ограничений должны предшествовать присваиванию новых значений?
4. От какой ошибки защищает копия списка в `Playlist.__init__`, а от какой — копия в `snapshot()`?
5. Что именно гарантирует свойство без setter? Можно ли всё равно намеренно изменить внутренний атрибут с `_`?

## Итоги

У каждого экземпляра своё состояние. Класс задаёт, как оно создаётся, какие операции разрешены и что делать при отказе. Проверяем обычный сценарий, границы, независимость экземпляров, общие ссылки и сохранность состояния после ошибки.

[Домашние задачи](tasks.md) продолжают работу с отдельными классами. Устройство программы из нескольких взаимодействующих классов — тема следующей лекции.